In [1]:
# ============================================================================
# 03_analysis_regressions_and_portfolios.ipynb
# US replication of the thesis' governance (Total Accrual) analysis:
#   (A) predictive regression of next-year GPOA on current TACC (Eq. 11 form),
#   (B) TACC-sorted quintile portfolios (annual July rebalance),
#   (C) CAPM / FF3 / Carhart alphas of the low-minus-high TACC hedge (Eq. 12-14).
# Grayscale seaborn figures are saved to ../results/figures at 600 dpi (png+pdf);
# tables are saved to ../results/tables.
# ============================================================================

In [2]:
# --- Imports, style, and helpers -----------------------------------------
import os
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

# Grayscale theme, no titles/captions, publication dpi.
sns.set_theme(style="whitegrid", palette="Greys_r")
plt.rcParams.update({"font.size": 11})

DATA_DIR = "../data"
FIG_DIR  = "../results/figures"
TAB_DIR  = "../results/tables"
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)

def savefig(fig, name):
    # Save both png and pdf at 600 dpi, no caption.
    for ext in ("png", "pdf"):
        fig.savefig(f"{FIG_DIR}/{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)

def winsorize(s, p=0.01):
    lo, hi = s.quantile(p), s.quantile(1 - p)
    return s.clip(lo, hi)

In [3]:
# --- Load data ------------------------------------------------------------
acc = pd.read_csv(f"{DATA_DIR}/accounting_panel.csv")
px  = pd.read_csv(f"{DATA_DIR}/monthly_prices.csv", index_col=0, parse_dates=True)
fac = pd.read_csv(f"{DATA_DIR}/ff_factors.csv", index_col=0, parse_dates=True)

# Normalize all monthly indices to month-start so prices, returns and factors
# align exactly (yfinance monthly stamps are month-start).
fac.index = fac.index.to_period("M").to_timestamp()

rets = px.pct_change().iloc[1:]
rets.index = rets.index.to_period("M").to_timestamp()
# Free price feeds contain occasional split/data-error spikes; clip them so a
# handful of bad ticks cannot dominate equal-weighted portfolio means.
rets = rets.clip(-0.9, 4.0)

In [4]:
# --- (A) Predictive regression: next-year GPOA on current TACC -----------
# Paper hypothesis H1-3: lower total accrual -> higher future performance.
# Controls: current GPOA. HAC (Newey-West, 3 lags) standard errors.
acc = acc.sort_values(["cik", "fyear"])
acc["gpoa_next"] = acc.groupby("cik")["gpoa"].shift(-1)

reg = acc.dropna(subset=["tacc", "gpoa", "gpoa_next"]).copy()
for c in ["tacc", "gpoa", "gpoa_next"]:
    reg[c] = winsorize(reg[c])

X = sm.add_constant(reg[["tacc", "gpoa"]])
m = sm.OLS(reg["gpoa_next"], X).fit(cov_type="HAC", cov_kwds={"maxlags": 3})

reg_tbl = pd.DataFrame({"coef": m.params, "t": m.tvalues, "p": m.pvalues}).round(4)
reg_tbl.to_csv(f"{TAB_DIR}/reg_gpoa_on_tacc.csv")
print("N =", int(m.nobs), "| TACC t =", round(m.tvalues['tacc'], 2))
reg_tbl

N = 2205 | TACC t = 1.14


,coef,t,p
const,0.3051,15.6952,0.0000
tacc,0.0080,1.1390,0.2547
gpoa,0.2969,7.5441,0.0000


In [5]:
# --- (B) TACC-sorted quintile portfolios ---------------------------------
# Convention matches the thesis (Table 12): P5 = LOWEST total accrual,
# P1 = HIGHEST. Firms are sorted each fiscal year; the portfolio is held from
# July of year t+1 to June of year t+2 (standard accounting-to-return lag).
acc["tacc_w"] = winsorize(acc["tacc"])

records = []
for fy in sorted(acc["fyear"].dropna().unique()):
    sub = acc[acc["fyear"] == fy].dropna(subset=["tacc_w"])
    sub = sub[sub["ticker"].isin(px.columns)]
    if len(sub) < 50:
        continue
    # qcut labels ascending (0=lowest value); invert so lowest TACC -> P5.
    sub = sub.copy()
    sub["q"] = 5 - pd.qcut(sub["tacc_w"], 5, labels=False, duplicates="drop")
    hold = pd.date_range(f"{int(fy)+1}-07-01", f"{int(fy)+2}-06-30", freq="MS")
    for q in range(1, 6):
        tks = [t for t in sub[sub["q"] == q]["ticker"].unique() if t in rets.columns]
        if not tks:
            continue
        pr = rets.reindex(hold)[tks].mean(axis=1)   # equal weight
        for dt, v in pr.items():
            if pd.notna(v):
                records.append((dt, q, v))

pdf = pd.DataFrame(records, columns=["date", "q", "ret"])
wide = pdf.pivot_table(index="date", columns="q", values="ret")
wide.columns = [f"P{c}" for c in wide.columns]
wide["P5_P1"] = wide["P5"] - wide["P1"]
wide = wide.dropna(how="all")
wide.to_csv(f"{TAB_DIR}/tacc_portfolio_returns.csv")
(wide[[f"P{i}" for i in range(1, 6)]].mean() * 100).round(3)

P1    1.688
P2    1.212
P3    1.399
P4    1.720
P5    3.949
dtype: float64

In [6]:
# --- (C) Factor-model alphas of the hedge portfolio ----------------------
# Regress the low-minus-high TACC return (P5-P1) on CAPM, FF3, and Carhart
# factors. A significant positive alpha means the accrual premium survives
# risk adjustment (paper Eq. 12-14). HAC (3 lags) standard errors.
d = wide.join(fac, how="inner").dropna(subset=["P5_P1"])
specs = {"CAPM": ["MktRF"],
         "FF3":  ["MktRF", "SMB", "HML"],
         "Carhart4": ["MktRF", "SMB", "HML", "WML"]}

rows = []
for name, cols in specs.items():
    mm = sm.OLS(d["P5_P1"], sm.add_constant(d[cols])).fit(
        cov_type="HAC", cov_kwds={"maxlags": 3})
    rows.append({"model": name,
                 "alpha(%)": round(mm.params["const"] * 100, 3),
                 "t(alpha)": round(mm.tvalues["const"], 2)})

alpha_tbl = pd.DataFrame(rows)
alpha_tbl.to_csv(f"{TAB_DIR}/tacc_hedge_alphas.csv", index=False)
alpha_tbl

,model,alpha(%),t(alpha)
0,CAPM,2.293,2.87
1,FF3,2.281,2.77
2,Carhart4,2.389,3.08


In [7]:
# --- Figure 1: mean monthly return by TACC quintile ----------------------
means = wide[[f"P{i}" for i in range(1, 6)]].mean() * 100
fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x=means.index, y=means.values, ax=ax, color="0.5", edgecolor="black")
ax.set_xlabel("Total accrual quintile (P1 = high, P5 = low)")
ax.set_ylabel("Mean monthly return (%)")
savefig(fig, "fig1_tacc_quintile_returns")

In [8]:
# --- Figure 2: cumulative growth, hedge vs market ------------------------
cum = (1 + d[["P5_P1", "MktRF"]]).cumprod()
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(cum.index, cum["P5_P1"], color="black", lw=1.6, label="Low-minus-high TACC (P5-P1)")
ax.plot(cum.index, cum["MktRF"], color="0.6", lw=1.6, ls="--", label="Market (MktRF)")
ax.set_xlabel("Date")
ax.set_ylabel("Cumulative growth of $1")
ax.legend(frameon=False)
savefig(fig, "fig2_hedge_cumulative")

In [9]:
# --- Figure 3: pooled TACC distribution ----------------------------------
fig, ax = plt.subplots(figsize=(6, 4))
sns.histplot(winsorize(acc["tacc"].dropna()), bins=60, color="0.5",
             edgecolor="black", ax=ax)
ax.set_xlabel("Total accruals (scaled by lagged assets)")
ax.set_ylabel("Frequency")
savefig(fig, "fig3_tacc_distribution")
print("Figures saved to", FIG_DIR)

Figures saved to ../results/figures
